# Experiment 001, Typo Robustness: Colab driver

Four steps: **bootstrap → configure → run → download**. Every pipeline stage
(data prep, generation, analysis, report) is driven by one command,
`tools/run_pipeline.py`, reading the single settings file
`configs/run_profile.yaml`. The full field list and a walkthrough of running
the same pipeline locally (no notebook) are in `RUNBOOK.md`.

Resume state is *only* the output directory named in the profile: re-running
this notebook after an interruption is safe, and a model with every row
already written is skipped without even loading it.


### 1. Bootstrap: clone, authenticate, install

One-time per Colab runtime. Set `BRANCH` below only if you're testing a
feature branch instead of `main`.


In [ ]:
BRANCH = "main"  # change only to test a feature branch

!git clone -b {BRANCH} https://github.com/natSegOS/glamor-research-onboarding.git
%cd glamor-research-onboarding/experiments/001_typo_robustness

import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded from Colab secret HF_TOKEN")
except Exception:
    # Not on Colab, or no HF_TOKEN secret configured: fall back to an
    # interactive login. Needed for gated repos (e.g. llama_1b).
    from huggingface_hub import notebook_login
    notebook_login()

!python tools/run_pipeline.py setup


### 2. Configure: the one cell you edit

Everything configurable lives here. `python tools/run_pipeline.py list-models`
(run it in a scratch cell) prints every available model key.

To parallelize across Google accounts, give each account's session a
`models` list containing just the one model that account should run — every
other field can stay identical.


In [ ]:
import yaml
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/glamor"
except ImportError:
    DRIVE_ROOT = "."  # not on Colab: keep everything local to the clone

profile = {
    # Which experiment config: configs/pilot.yaml, configs/rehearsal.yaml,
    # or configs/main.yaml (the confirmatory run; requires pinned revisions).
    "experiment_config": "configs/rehearsal.yaml",

    # Which models to run this session (see list-models for every key).
    "models": [
        "qwen_1b5_pilot", "llama_1b", "llama_3b",
        "qwen_7b_awq", "llama_8b_awq",
    ],

    # Where results go. A Drive path survives a runtime restart.
    "output_root": f"{DRIVE_ROOT}/results/rehearsal_v2",
    "analysis_dir": "analysis/rehearsal_v2",

    # Reuse the committed data/items + dictionary instead of rebuilding them.
    # Confirmatory runs (main.yaml) always rebuild regardless of this flag.
    "rebuild_data": False,

    # Don't even load a model whose rows are all already written.
    "skip_if_complete": True,
}

Path("configs/run_profile.yaml").write_text(yaml.safe_dump(profile, sort_keys=False))
print(yaml.safe_dump(profile, sort_keys=False))


### 3. Run

Prints a plan (what will run, what's already complete, whether data is being
reused) before touching any GPU, then runs data prep → generation → analysis
→ report. A model that fails (OOM, gated access) is recorded and skipped;
the rest of the roster continues.


In [ ]:
!python tools/run_pipeline.py all


### 4. Download results


In [ ]:
import zipfile
from pathlib import Path

output_root = Path(profile["output_root"])
archive_path = Path("rehearsal_results.zip")

with zipfile.ZipFile(archive_path, "w") as archive:
    for path in Path(profile["analysis_dir"]).rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to("."))
    for path in output_root.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(output_root.parent))

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print(f"Saved to {archive_path.resolve()}")
